## **Finetune BERT Encoder for Text Classification**

The framework to building NLP systems using large language models is simple:

*   Select a pretrained language model that generates word embeddings
*   Attach a classification head appropriate for the downstream classification task - this could be Sequence Classification or Token Classification. If we are working on sequence generation, we pass the encoder embeddings to a Decoder for language generation.
*   Train the entire model with a small set of labelled examples that are suitable for the downstream task. This process is called fine-tuning. Here the encoder is trained alongside the neural engine that generates final predictions.
*   The embeddings generated in this manner are dynamic. Word representations in this setup alter with the text in which the words appear themselves. Hence these embeddings are called **contextual embeddings**.

In this notebook, we want to test the effectiveness of BERT encoders for Sentiment Classification.

#### **0. Housekeeping Steps**

Let us perform the necessary housekeeping steps before procedding further with the Machine Learning task at hand. We probably need to install some packages before we can import them.

In [1]:
!pip install transformers
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 24.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pla

In addition we need to link our Colab notebook to our Google Drive so we can both save and load our necessary models and data. To do this we need to mount our Google Drive locally. Please be mindful of repeating this step.

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
#sanity check to ensure drive was properly mounted
!ls /content/gdrive/MyDrive/DSSI-2024-Week2-internal/notebooks

 Day1-rough.ipynb
 e2e-dataset
'Homework1: Text Classification using Pre-trained Embeddings.ipynb'
'Homework2: Text Classification using Finetuning.ipynb'
'Lab 1+2: PyTorch+Neural-Networks-for-Beginners.ipynb'
'Lab 3: Classification for NLP Demo.ipynb'
'Lab 4: Hugging_Face_Transformers_Tutorial'
'Lab 5: Token Classification with HuggingFace Transformers'
'Overview of Colaboratory Features'
 python-dictionary-review.ipynb
 sample_hf_trainer
 sample_pte_trainer
'Solved-Homework1: Text Classification using Pre-trained Embeddings.ipynb'
'Solved-Homework2: Text Classification using Finetuning.ipynb'
 sst-model


**NOTE:** In order to load and save models we need to set the model path. First, create a folder named `sst-model` in this path where you can save your best performing model.

In [4]:
model_path = "/content/gdrive/MyDrive/DSSI-2024-Week2-internal/notebooks/sst-model/"

#### **1. Imports**

Let us start with all the necessary imports

In [5]:
from collections import defaultdict, Counter
import json
import numpy as np
import torch

from matplotlib import pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, DatasetDict
from torch.utils.data import DataLoader

#### **2. Loading the data**

We start with loading a dataset from the Hugging Face hub. Note that for our projects we will have to work with custom datasets and that would require extending the `Dataset` class to a custom Dataset subclass specific to your dataset. However, in the interest of time, we will simply download an existing dataset from the hub that is already in the desired format.

For the purpose of this homework, we want to use the **Stanford Sentiment Treebank Dataset** for sentiment classification.

In [6]:
#Download the Stanford Sentiment Treebank Dataset

dataset_name = "stanfordnlp/sst2"
sst_dataset = load_dataset(dataset_name)

sst_dataset

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

#### **3. Dataset Preprocessing**

Next we tokenize and prepare the data. For our choice of model and associated tokenizer we want to use `bert-base-uncased` from the Hugging Face model hub.

As you might remember, the tokenizer executes the following steps:


1.   Split text into tokens and convert them into word ids
2.   Add special tokens like [CLS] and [SEP]
3.   Padding the text so all inputs are of the same length
4.   Apply truncation when needed by setting a max length for sequences.



In [7]:
# from transformers import BertTokenizer, BertModel, BertConfig, BertForSequenceClassification
# name = "google-bert/bert-base-cased"

from transformers import DistilBertConfig, DistilBertTokenizer, DistilBertForSequenceClassification, DistilBertModel
name = "distilbert/distilbert-base-cased"
#Check if tokenizer was properly loaded by using on a test sentence
tokenizer = DistilBertTokenizer.from_pretrained(name)

sample_input = "We want to use a pretrained tokenizer."
tokenized_inputs = tokenizer(sample_input,
                             return_tensors="pt",
                             padding=True,
                             truncation=True,
                             max_length=128)
print(tokenized_inputs["input_ids"])

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tensor([[  101,  1284,  1328,  1106,  1329,   170,  3073,  4487,  9044, 22559,
         17260,   119,   102]])


Now that our tokenizer has been loaded let us use it to tokenize our entire dataset. We will use the function that we use to test the tokenizer on a single input. We will also split our data into batches of 128.

In [8]:
tokenized_sst_dataset = sst_dataset.map(
    lambda example: tokenizer(example['sentence'], padding="max_length",
    truncation=True, max_length=64)
)

tokenized_sst_dataset = tokenized_sst_dataset.remove_columns(['idx', 'sentence'])
tokenized_sst_dataset = tokenized_sst_dataset.rename_column("label", "labels")
tokenized_sst_dataset.set_format("torch")

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [9]:
#lets check our tokenization for a few samples
tokenized_sst_dataset['train'][0:2]

{'labels': tensor([0, 0]),
 'input_ids': tensor([[  101,  4750,  1207,  3318,  5266,  1121,  1103, 22467,  2338,   102,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0],
         [  101,  2515,  1185, 20787,   117,  1178,  5530,  1174, 21102,  1116,
            102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,    

#### **4. Using DataLoader to batchify data**

Make sure to send your datasets to the Dataloader in order to segment your dataset into batches. Remember that we need batches to run our iterative optimization procedure which is typically some form of Mini-batch Gradient Descent.

In the interest of time we want to finetune our model on a sample of the training set with 2048 records instead of the entire 62K sample size.

In [10]:
train_dataset = tokenized_sst_dataset['train'].shuffle(seed=1111).select(range(2048))
train_dataloader = DataLoader(train_dataset, batch_size=16)
eval_dataloader = DataLoader(tokenized_sst_dataset['validation'], batch_size=16)

#### **5. Training and Validation**

We have now gone through all the required preprocessing to prep the data for training. Instead of the Trainer module, it will be a good practice to initially write our own training loops so that we are mindful of all the steps that required for training neural networks.

Other than our training and validation data we need to select:


*   An optimizer to run backpropagation
*   A scheduler that sets a protocol for parameter updates at the end of a batch

We would also like to set a seed at the start of computation. This ensures that we are able to generate reproducicble results across multiple training sessions.

We run validation at the end of each epoch.

***At the end of this step we want to report the best validation loss obtained during training. We also want to save the model corresponding to the epoch that reported the best validation loss.***


In [12]:
from transformers import get_linear_schedule_with_warmup
from tqdm.notebook import tqdm
from torch.optim import AdamW
from transformers import set_seed

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#comes with a loss function
model = DistilBertForSequenceClassification.from_pretrained(name, num_labels=2).to(device)

num_epochs = 4
num_training_steps = len(train_dataloader)
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
lr_scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

best_val_loss = float("inf")
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    # training
    model.train()
    training_losses = []
    for batch_i, batch in enumerate(train_dataloader):

        optimizer.zero_grad()

        # copy input to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # output = model(**batch)
        output = model(input_ids, attention_mask=attention_mask, labels=labels)
        training_loss = output.loss
        training_losses.append(training_loss.item())

        #backprop and update params by taking an optimization step
        output.loss.backward()
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)
    print("Mean Training Loss", np.mean(training_losses))

    # validation
    val_loss = 0
    model.eval() #important to call because we dont want to collect gradients
    for batch_i, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            # copy input to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            # output = model(**batch)
            output = model(input_ids, attention_mask=attention_mask, labels=labels)
        val_loss += output.loss

    avg_val_loss = val_loss / len(eval_dataloader)
    print(f"Validation loss: {avg_val_loss}")
    if avg_val_loss < best_val_loss:
        print("Saving checkpoint!")
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            # 'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
            },
            f"{model_path}epoch_{epoch}.pt"
        )
    print()

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/128 [00:00<?, ?it/s]

Mean Training Loss 0.4326627143891528
Validation loss: 0.3486649692058563
Saving checkpoint!

Mean Training Loss 0.2396822913433425
Validation loss: 0.3486649692058563

Mean Training Loss 0.23544734116876498
Validation loss: 0.3486649692058563

Mean Training Loss 0.23877627128968015
Validation loss: 0.3486649692058563



#### **6. Evaluate your model on Test Data**

Now we use our finetuned model to evaluate the test set. We use performance metrics from `sklearn.metrics` to test the effectiveness of our model on unseen test data.

In order to do that, run the finetuned model you have just saved on your test data and report the following performance metrics:



*   Accuracy
*   F1 Score



In [13]:
from sklearn.metrics import accuracy_score, f1_score

In [14]:
eval_dataloader = DataLoader(tokenized_sst_dataset['validation'], batch_size=len(tokenized_sst_dataset['validation']))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.eval()
test_batch_logits = []
y_true = []
for batch_i, batch in enumerate(eval_dataloader):
    with torch.no_grad():
        # copy input to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().detach().numpy()
        # output = model(**batch)
        output = model(input_ids, attention_mask=attention_mask)
        test_batch_logits.append(output.logits)
        y_true.extend(labels)

In [15]:
print(len(test_batch_logits),len(eval_dataloader))
test_logits = torch.cat(test_batch_logits, dim=0)

#sanity check -> dimension 0 of your logits tensor should be same as the size of the test dataset
print(test_logits.shape,len(tokenized_sst_dataset['validation']),len(y_true))

1 1
torch.Size([872, 2]) 872 872


In [16]:
#Convert the logits to predicted labels
y_pred = torch.argmax(test_logits, dim = 1).cpu().numpy()
print(y_true[:10])
print(y_pred[:10])

#sanity check: should have as many predictions as labels
assert len(y_pred)==len(y_true)

[np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0)]
[1 0 1 1 0 1 0 0 1 0]


In [17]:
print('F1 Score:',f1_score(y_true, y_pred))
print('Accuracy Score:',accuracy_score(y_true, y_pred))

F1 Score: 0.8525714285714285
Accuracy Score: 0.8520642201834863
